# 02. Modelagem Preditiva (regressão e classificação)
## Tech Challenge Fase 1, Case NPS Preditivo

Estas são as fases de Modeling e Evaluation do CRISP-DM, que cobrem o requisito 4
(opcional). A ideia é prever o NPS antes da pesquisa usando só dados operacionais
(pedido, logística, atendimento), para a empresa agir de forma preventiva.

Faço de dois jeitos: uma regressão para estimar a nota de NPS (0 a 10) e uma
classificação para sinalizar se o cliente vai ser detrator (nota até 6).

In [1]:
# Localiza a pasta src/ subindo a partir do diretorio atual, para o notebook
# funcionar tanto rodando de notebooks/ quanto da raiz do projeto.
import sys
from pathlib import Path

def _find_src(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "src" / "config.py").exists():
            return cand / "src"
    raise FileNotFoundError("Pasta src/ nao encontrada a partir de " + str(p))

SRC = _find_src()
sys.path.insert(0, str(SRC))
print("src em:", SRC)

src em: C:\workspace-fiap\tech-challenge-fase1-nps-preditivo\src


In [2]:
import json
import pandas as pd
import config
import modeling
from data_preparation import load_processed

df = load_processed()
df.shape

(2500, 22)

## 1. Seleção de variáveis

Uso só variáveis operacionais, disponíveis antes da pesquisa. Excluo de propósito:

| Variável | Motivo |
|---|---|
| repeat_purchase_30d | Vazamento temporal, só é conhecida 30 dias depois |
| csat_internal_score | Proxy de satisfação, prever satisfação com satisfação é circular |

In [3]:
print("Features usadas (operacionais):")
for f in config.OPERATIONAL_FEATURES:
    print("  -", f)
print("\nExcluidas do modelo:", config.EXCLUDED_FROM_PRIMARY_MODEL)

Features usadas (operacionais):
  - order_value
  - items_quantity
  - discount_value
  - payment_installments
  - delivery_time_days
  - delivery_delay_days
  - freight_value
  - delivery_attempts
  - customer_service_contacts
  - resolution_time_days
  - complaints_count
  - customer_age
  - customer_tenure_months
  - customer_region

Excluidas do modelo: ['repeat_purchase_30d', 'csat_internal_score']


## 2. Treino do pipeline

O modeling.run() faz o split treino/teste (80/20, estratificado), o pré-processamento
(padronização mais one-hot), treina vários modelos, avalia, e salva os modelos
(.joblib), as figuras e o models/metrics.json.

In [4]:
metrics = modeling.run()

MODELAGEM - resumo
Regressao  -> melhor: Regressao Linear | R2=0.545 | MAE=1.354
Classific. -> melhor: Regressao Logistica | AUC=0.905 | recall_detrator=0.846 | F1=0.889
Vazamento  -> AUC: operacional=0.898 vs com suspeitas=0.944 | R2: operacional=0.548 vs com suspeitas=0.645

Modelos e metrics.json salvos em C:\workspace-fiap\tech-challenge-fase1-nps-preditivo\models


## 3. Regressão, estimar a nota de NPS
Comparação dos modelos no conjunto de teste:

In [5]:
reg = metrics["regressao"]
print("Melhor modelo:", reg["melhor_modelo"], "| R2 (cross-val):", reg["cv_r2_media"])
pd.DataFrame(reg["modelos"]).T

Melhor modelo: Regressao Linear | R2 (cross-val): 0.552


,MAE,RMSE,R2
Baseline (media),2.086,2.549,-0.000
Regressao Linear,1.354,1.720,0.545
Random Forest,1.423,1.780,0.512
Gradient Boosting,1.371,1.729,0.540


O modelo erra em média cerca de 1,35 ponto na escala de 0 a 10 e explica perto de 55%
da variação do NPS só com dados operacionais, o que é um resultado sólido para uso
preventivo.

![Previsto vs real](../reports/figures/10_regressao_pred_vs_real.png)

## 4. Classificação, sinalizar detratores
O foco é o recall de detratores, para não deixar passar quem está em risco.

In [6]:
clf = metrics["classificacao"]
print("Melhor modelo:", clf["melhor_modelo"])
pd.DataFrame(clf["modelos"]).T

Melhor modelo: Regressao Logistica


,acuracia,acuracia_balanceada,precisao_detrator,recall_detrator,f1_detrator,roc_auc
Baseline (mais frequente),0.790,0.500,0.790,1.000,0.883,0.500
Regressao Logistica,0.834,0.818,0.938,0.846,0.889,0.905
Random Forest,0.854,0.705,0.868,0.962,0.912,0.898


In [7]:
print("Matriz de confusao:")
print(json.dumps(clf["matriz_confusao"], indent=2, ensure_ascii=False))

Matriz de confusao:
{
  "verdadeiro_nao_detrator": 83,
  "falso_detrator": 22,
  "falso_nao_detrator": 61,
  "verdadeiro_detrator": 334
}


![Matriz de confusão](../reports/figures/11_classificacao_matriz_confusao.png)
![Curva ROC](../reports/figures/13_curva_roc.png)

## 5. Quais variáveis o modelo mais usa
Importância por permutação, no modelo de classificação.

In [8]:
pd.Series(clf["importancia_variaveis"]).head(8)

delivery_delay_days          0.1745
complaints_count             0.1719
resolution_time_days         0.0268
order_value                  0.0006
freight_value                0.0006
customer_age                 0.0002
customer_service_contacts    0.0002
customer_tenure_months       0.0002
dtype: float64

Confirma a EDA: atraso na entrega e reclamações dominam.

![Importância](../reports/figures/12_importancia_variaveis.png)

## 6. Por que excluir as variáveis suspeitas

Incluir o repeat_purchase_30d e o csat_internal_score melhora as métricas, mas elas não
estão disponíveis na hora da previsão e são proxies de satisfação. Mostrar isso
justifica mantê-las de fora.

In [9]:
leak = metrics["demonstracao_vazamento"]
print("Classificacao (AUC):", leak["classificacao_auc"])
print("Regressao (R2)     :", leak["regressao_r2"])
print("\n", leak["leitura"])

Classificacao (AUC): {'operacional': 0.898, 'com_vazamento': 0.944}
Regressao (R2)     : {'operacional': 0.548, 'com_vazamento': 0.645}

 Incluir as variaveis suspeitas eleva AUC e R2, mas elas nao estao disponiveis antes da pesquisa (vazamento) e sao proxies de satisfacao (circularidade). Por isso ficam fora do modelo.


## 7. Como usar na prática (Deployment)

A cada pedido, o modelo calcula a probabilidade de detrator com dados operacionais que
já existem. A partir disso dá para ter gatilhos de ação: pedido com atraso de 2 dias ou
mais, ou com 2 reclamações ou mais, entra numa fila de recuperação proativa (contato,
compensação, priorização). A importância das variáveis orienta onde investir primeiro,
que é pontualidade da entrega e redução de reclamações. Em produção, valeria acompanhar
o NPS previsto contra o realizado e retreinar de tempos em tempos.

Limitações: a base é sintética, então os padrões precisam ser confirmados com dados
reais; correlação não é causa, então o ideal é validar com teste A/B; e o modelo
prioriza, não substitui o julgamento das áreas.